# SatQuery AI — LoRA fine-tune on Kaggle (Qwen2.5-VL-3B)

Day 3-4 of the sprint. Trains a QLoRA adapter on the balanced RSVQA-LR set,
merges it into an fp16 base, and converts the result to 4-bit MLX — all in
this notebook — because a `peft`/CUDA adapter cannot load into MLX on the
Mac (different tensor layout, different key prefixes). See
`scripts/README_LORA.md` in the repo for the full writeup of that trap.

**What comes out the other end:** an `mlx-4bit/` model directory
(~2 GB), downloadable from the Output tab, that drops straight into

```bash
SATQUERY_VLM_BACKEND=mlx SATQUERY_MLX_MODEL_ID=/path/to/mlx-4bit uvicorn app.main:app
```

**Hardware:** Kaggle's 2× T4 (SM 7.5). This notebook deliberately uses
**one** T4, not both — `device_map="auto"` across two T4s is naive
pipeline parallelism, not a speedup, and pinning to one GPU is simpler and
just as fast for a 3B model. T4 has no bf16 and no flash-attention-2:
every load below is `fp16`, `attn_implementation="sdpa"`.

**Budget:** Kaggle gives 30 GPU-hours/week. 3 epochs on the ~33.5k-row
balanced set is the target; `SMOKE_TEST = True` below runs a ~300-row,
1-epoch dry run first so a broken cell fails in 10 minutes, not 3 hours.


## Before you run this — checklist

**1. On the Mac, build the balanced dataset (if you haven't already):**

```bash
cd satquery-ai/backend
.venv/bin/python scripts/balance_dataset.py   # writes data/train_balanced.jsonl
```

**2. Package a Kaggle Dataset.** The JSONL stores absolute Mac paths
(`/Users/.../Images_LR/92.tif`) which obviously don't exist on Kaggle — this
notebook remaps every image reference to `IMAGES_DIR` by **filename only**,
so the upload just needs the three things below, flat, no path-preserving
tricks required:

```bash
mkdir -p /tmp/kaggle_upload/Images_LR
cp data/train_balanced.jsonl data/train_val.jsonl /tmp/kaggle_upload/
cp ~/data/RSVQA-LR/Images_LR/*.tif /tmp/kaggle_upload/Images_LR/
```

Go to **kaggle.com/datasets → New Dataset**, upload the contents of
`/tmp/kaggle_upload/`, name it (e.g. `rsvqa-lr-balanced`), create it private.

**3. Attach it to this notebook.** Notebook → **Add Input** → your dataset.
Set `KAGGLE_DATASET_SLUG` in the next cell to match the folder name shown
under `/kaggle/input/`.

**4. Notebook settings** (right sidebar):
- **Accelerator:** GPU T4 × 2
- **Internet:** ON (pulls the base model from the Hugging Face Hub)
- **Persistence:** Files only is fine — the final artifact is tarred and
  saved under `/kaggle/working` for the Output tab regardless.

**5. If `Qwen/Qwen2.5-VL-3B-Instruct` requires a Hub login** (gated repos do):
**Add-ons → Secrets** → add one named `HF_TOKEN` with your Hugging Face
access token. The notebook reads it automatically if present; if the repo
isn't gated, it's simply unused.

**6. To actually execute all of this on Kaggle's GPUs** (editing a notebook
does not run it): **Save Version → Save & Run All (Commit)**. Come back when
it finishes, open that version, and download from its **Output** tab.


In [ ]:
# CUDA_VISIBLE_DEVICES must be set before torch (or anything that imports
# torch) is ever imported in this process - once CUDA initialises it locks
# in which devices are visible for the life of the kernel. This is the one
# line standing between us and accidentally doing naive 2-GPU pipeline
# parallelism through device_map="auto".
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# ---- edit me -----------------------------------------------------------
KAGGLE_DATASET_SLUG = "rsvqa-lr-balanced"   # must match your Add Input name
SMOKE_TEST = True                            # flip to False for the real run
# --------------------------------------------------------------------------

from pathlib import Path

INPUT_DIR = Path("/kaggle/input") / KAGGLE_DATASET_SLUG
IMAGES_DIR = INPUT_DIR / "Images_LR"
TRAIN_JSONL = INPUT_DIR / "train_balanced.jsonl"
VAL_JSONL = INPUT_DIR / "train_val.jsonl"

WORK_DIR = Path("/kaggle/working")
OUT_DIR = WORK_DIR / "out"
ADAPTER_DIR = OUT_DIR / "adapter"
MERGED_DIR = OUT_DIR / "merged"
MLX_DIR = OUT_DIR / "mlx-4bit"
for d in (OUT_DIR, ADAPTER_DIR, MERGED_DIR, MLX_DIR):
    d.mkdir(parents=True, exist_ok=True)

# %%writefile does NOT create parent directories, and Kaggle's working dir
# starts empty - without this the cuda_sanity cell dies with FileNotFoundError.
Path("scripts").mkdir(exist_ok=True)

# The HF (non-quantized-repo) counterpart of the Mac's
# mlx-community/Qwen2.5-VL-3B-Instruct-4bit - same base model, so the LoRA
# we train here is meaningful once merged and re-quantized on the Mac side.
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

# MUST match app/config.py's serving values (SATQUERY_MAX_PIXELS /
# SATQUERY_MIN_PIXELS on the Mac). This is not a speed knob - it changes
# what the model is trained to see. A mismatch degrades answers with no
# error at all. 200704 = 256*28*28, the Mac's recommended value (not the
# 768*28*28 CUDA-lane default in app/config.py).
MAX_PIXELS = 200704
MIN_PIXELS = 64 * 28 * 28

LEARNING_RATE = 5e-5   # not 1e-4 - fp16 + LoRA on Qwen2-VL NaNs at the higher LR
NUM_EPOCHS = 1 if SMOKE_TEST else 3
TRAIN_SUBSET = 300 if SMOKE_TEST else None
VAL_SUBSET = 60 if SMOKE_TEST else None
PER_DEVICE_BATCH = 2
GRAD_ACCUM_STEPS = 4   # effective batch size 8

print(f"SMOKE_TEST      = {SMOKE_TEST}")
print(f"INPUT_DIR       = {INPUT_DIR} (exists={INPUT_DIR.exists()})")
print(f"MODEL_ID        = {MODEL_ID}")
print(f"MAX_PIXELS      = {MAX_PIXELS}  MIN_PIXELS = {MIN_PIXELS}")
print(f"NUM_EPOCHS      = {NUM_EPOCHS}")

# Fail here, loudly, rather than six cells later inside the data loader.
# The usual causes, in order of likelihood: the dataset was attached after
# this session started (restart the kernel), or Kaggle mounted it under a
# different folder name than KAGGLE_DATASET_SLUG.
if not INPUT_DIR.exists():
    available = sorted(p.name for p in Path("/kaggle/input").iterdir()) \
        if Path("/kaggle/input").exists() else []
    raise SystemExit(
        f"\nSTOP: {INPUT_DIR} does not exist.\n"
        f"Datasets currently mounted: {available or 'NONE'}\n\n"
        + (
            f"-> set KAGGLE_DATASET_SLUG to one of {available} above and re-run "
            "this cell.\n"
            if available else
            "-> no dataset is attached. Right sidebar > Add Input > add your\n"
            "   dataset, then hit the restart button and run this cell again.\n"
        )
    )
print("\ndataset OK")


In [ ]:
# Packages Kaggle's base image does not ship. Deliberately NOT touching
# torch itself - Kaggle pins it to match the preinstalled CUDA driver, and
# forcing an upgrade is how you end up with a torch that can't see the GPU.
# pillow is capped for the same reason: Kaggle's torchvision is built against
# pillow<12, transformers imports torchvision, and an unpinned upgrade breaks
# `from transformers import ...` with an ImportError deep inside PIL.
%pip install -q -U "transformers>=4.49" "accelerate>=1.0" "peft>=0.13" \
    "bitsandbytes>=0.43" "pillow<12"

# Kaggle ships torchao, and peft's LoRA dispatcher probes it on every
# inject_adapter call. A version peft dislikes raises ImportError from inside
# PeftModel.from_pretrained - after training has already run. We quantise with
# bitsandbytes, never torchao, so removing it takes peft off that path.
%pip uninstall -q -y torchao

print("done")


## Step 1 — host sanity check

Run `scripts/cuda_sanity.py` (same file as the repo's `backend/scripts/`)
before touching the model. It fails loudly on exactly the two traps that
otherwise waste GPU-hours: bf16 on a card that doesn't have it, and
flash-attention-2 on pre-Ampere silicon.


In [ ]:
%%writefile scripts/cuda_sanity.py
#!/usr/bin/env python3
"""Day 1: CUDA sanity check. Run on the 4060 AND in the Kaggle/Colab notebook.

Prints one report and exits non-zero if the host cannot train or serve, so it
can gate a CI step or a notebook cell.
"""
from __future__ import annotations

import sys


def main() -> int:
    ok = True
    print("=" * 62)
    print("SatQuery AI - CUDA sanity")
    print("=" * 62)

    try:
        import torch
    except ImportError:
        print("FAIL  torch not installed")
        return 1

    print(f"torch            {torch.__version__}")
    print(f"cuda build       {torch.version.cuda}")

    if not torch.cuda.is_available():
        print("FAIL  no CUDA device visible")
        print("      -> CPU host: run the API with SATQUERY_VLM_BACKEND=mock")
        return 1

    n = torch.cuda.device_count()
    print(f"devices          {n}")
    for i in range(n):
        prop = torch.cuda.get_device_properties(i)
        cap = f"{prop.major}.{prop.minor}"
        print(f"  [{i}] {prop.name}  {prop.total_memory / 1024**3:.1f} GB  SM {cap}")

    major = torch.cuda.get_device_capability()[0]
    if major >= 8:
        print("dtype            bf16 supported (Ampere+)")
    else:
        print("dtype            fp16 ONLY - this GPU has no bf16 (pre-Ampere, e.g. T4)")
        print("                 -> set fp16=True, bf16=False in TrainingArguments")

    try:
        import flash_attn  # noqa: F401

        has_fa = True
    except ImportError:
        has_fa = False
    attn = "flash_attention_2" if (major >= 8 and has_fa) else "sdpa"
    print(f"attn impl        {attn}")
    if major < 8:
        print("                 -> flash_attention_2 will RAISE on this GPU. Do not")
        print("                    copy it from a Qwen2-VL snippet; use sdpa.")

    try:
        import bitsandbytes as bnb

        print(f"bitsandbytes     {bnb.__version__}")
    except ImportError:
        print("FAIL  bitsandbytes missing - no 4-bit quantisation")
        ok = False

    for mod in ("transformers", "peft", "accelerate"):
        try:
            print(f"{mod:16} {__import__(mod).__version__}")
        except ImportError:
            print(f"FAIL  {mod} missing")
            ok = False

    # Real allocation + matmul, not just a version check.
    try:
        torch.cuda.reset_peak_memory_stats()
        a = torch.randn(4096, 4096, device="cuda", dtype=torch.float16)
        (a @ a).sum().item()
        torch.cuda.synchronize()
        peak = torch.cuda.max_memory_allocated() / 1024**3
        del a
        torch.cuda.empty_cache()
        print(f"matmul smoke     OK (peak {peak:.2f} GB, freed)")
    except Exception as exc:
        print(f"FAIL  matmul: {exc}")
        ok = False

    free, total = torch.cuda.mem_get_info()
    print(f"free VRAM        {free / 1024**3:.2f} / {total / 1024**3:.2f} GB")
    if free / 1024**3 < 5.0:
        print("WARN  under 5 GB free - close other processes before profiling")

    print("=" * 62)
    print("PASS" if ok else "FAIL")
    return 0 if ok else 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
import subprocess

result = subprocess.run(["python", "scripts/cuda_sanity.py"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, (
    "cuda_sanity.py failed - fix the host before spending GPU-hours on training"
)


## Step 2 — load the balanced dataset

Remap every `"image"` field to `IMAGES_DIR` by filename — the JSONL was
built on the Mac and carries Mac-only absolute paths.


In [ ]:
# Kaggle does not always mount a dataset at /kaggle/input/<the name you typed>,
# and a zip upload can leave the contents one directory deeper than expected.
# Rather than hand-editing KAGGLE_DATASET_SLUG until it matches, locate the
# three things we actually need and rebind the paths to wherever they landed.
train_hits = sorted(Path("/kaggle/input").rglob("train_balanced.jsonl"))
val_hits = sorted(Path("/kaggle/input").rglob("train_val.jsonl"))
img_hits = sorted(p for p in Path("/kaggle/input").rglob("Images_LR") if p.is_dir())

assert train_hits, "train_balanced.jsonl not found anywhere under /kaggle/input"
assert val_hits, "train_val.jsonl not found anywhere under /kaggle/input"
assert img_hits, "no Images_LR directory found under /kaggle/input"

TRAIN_JSONL = train_hits[0]
VAL_JSONL = val_hits[0]
IMAGES_DIR = img_hits[0]
INPUT_DIR = TRAIN_JSONL.parent

n_tif = len(list(IMAGES_DIR.glob("*.tif")))
print(f"INPUT_DIR   = {INPUT_DIR}")
print(f"TRAIN_JSONL = {TRAIN_JSONL}")
print(f"VAL_JSONL   = {VAL_JSONL}")
print(f"IMAGES_DIR  = {IMAGES_DIR}  ({n_tif} tif)")
assert n_tif > 500, f"only {n_tif} images - the upload is incomplete"


In [ ]:
import copy
import json


def load_jsonl(path: Path) -> list[dict]:
    with path.open() as fh:
        return [json.loads(line) for line in fh if line.strip()]


def remap_images(records: list[dict], images_dir: Path) -> list[dict]:
    out = []
    for r in records:
        r = copy.deepcopy(r)
        content = r["messages"][0]["content"]
        for item in content:
            if item.get("type") == "image":
                item["image"] = str(images_dir / Path(item["image"]).name)
        out.append(r)
    return out


train_records = remap_images(load_jsonl(TRAIN_JSONL), IMAGES_DIR)
val_records = remap_images(load_jsonl(VAL_JSONL), IMAGES_DIR)

if TRAIN_SUBSET:
    train_records = train_records[:TRAIN_SUBSET]
if VAL_SUBSET:
    val_records = val_records[:VAL_SUBSET]

print(f"train rows = {len(train_records)}   val rows = {len(val_records)}")

# Fail fast, not 40 minutes into an epoch: every image referenced by the
# first N rows must actually exist under IMAGES_DIR.
missing = [
    r["messages"][0]["content"][0]["image"]
    for r in train_records[:200]
    if not Path(r["messages"][0]["content"][0]["image"]).exists()
]
assert not missing, f"{len(missing)} images not found, e.g. {missing[:3]} - check IMAGES_DIR / the dataset upload"
print("image paths OK (spot-checked first 200 rows)")
print(train_records[0])


## Step 3 — load the base model in 4-bit (QLoRA) and the processor

Single GPU (`device_map={"": 0}`), `fp16` compute dtype, `sdpa` attention —
the three non-negotiables for a T4. `AutoProcessor` gets the same
`min_pixels`/`max_pixels` the Mac serves at, mirroring
`app/services/vlm.py`'s `LocalQwen2VL` construction exactly.


In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

# NOT torch.cuda.is_bf16_supported() - that returns True on a T4 because recent
# PyTorch counts software emulation as "supported". Compute capability is the
# real answer, and it is what scripts/cuda_sanity.py already checks.
cap = torch.cuda.get_device_capability()
assert cap[0] < 8, (
    f"this GPU is SM {cap[0]}.{cap[1]} (Ampere+) - you are not on a T4. "
    "Double check the accelerator setting before proceeding."
)
print(f"SM {cap[0]}.{cap[1]} - fp16 path confirmed")

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass  # no secret configured - fine for a non-gated repo

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # NOT bfloat16 - T4 is SM 7.5
    bnb_4bit_use_double_quant=True,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map={"": 0},          # one GPU, not "auto" across both T4s
    attn_implementation="sdpa",  # flash_attention_2 raises on SM < 8.0
    torch_dtype=torch.float16,
    token=hf_token,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS, token=hf_token,
)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print("model + processor loaded")
print(f"device: {model.device}")


## Step 4 — attach LoRA

`target_modules=["q_proj", "v_proj"]` hits the language decoder's attention
projections. The vision tower uses a fused `qkv` linear, not separate
`q_proj`/`v_proj`/`k_proj`, so this LoRA config never touches it — that's
checked explicitly below by printing every module name the adapter matched.

fp16 + LoRA on Qwen2-VL can produce NaN losses; keeping the LoRA weights in
fp32 (while the frozen base stays 4-bit/fp16) is the fix, on top of the
lower learning rate.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False  # incompatible with gradient checkpointing
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

# Belt and suspenders: force every LoRA param to fp32 regardless of what
# this peft version defaults to for a quantized base.
n_cast = 0
for name, param in model.named_parameters():
    if "lora_" in name and param.dtype != torch.float32:
        param.data = param.data.float()
        n_cast += 1
print(f"cast {n_cast} LoRA tensors to fp32")

model.print_trainable_parameters()

matched = sorted({n for n, _ in model.named_modules() if "lora_A" in n or "lora_B" in n})
print(f"\nLoRA attached to {len(matched)} modules, e.g.:")
for m in matched[:6]:
    print(" ", m)
assert all("visual" not in m for m in matched), (
    "LoRA matched something inside the vision tower - target_modules is wrong"
)


## Step 5 — dataset, collator, training args

Labels are masked to -100 over the prompt (image + question) so the loss is
computed on the assistant's answer tokens only. `image_grid_thw` and
`pixel_values` are batched Qwen2-VL's way: patches concatenated along dim 0,
grid shapes stacked — the same tensors `LocalQwen2VL._build` produces per
example in `app/services/vlm.py`, just batched here.


In [ ]:
from PIL import Image
from torch.utils.data import Dataset


class VQADataset(Dataset):
    def __init__(self, records, processor):
        self.records = records
        self.processor = processor

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        image_path = record["messages"][0]["content"][0]["image"]
        question = record["messages"][0]["content"][1]["text"]
        answer = record["messages"][1]["content"][0]["text"]
        image = Image.open(image_path).convert("RGB")

        user_turn = [{"role": "user", "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": question},
        ]}]
        prompt_text = self.processor.apply_chat_template(
            user_turn, tokenize=False, add_generation_prompt=True
        )
        full_text = self.processor.apply_chat_template(
            user_turn + [{"role": "assistant", "content": [{"type": "text", "text": answer}]}],
            tokenize=False, add_generation_prompt=False,
        )

        prompt_ids = self.processor(text=[prompt_text], images=[image], return_tensors="pt")
        full = self.processor(text=[full_text], images=[image], return_tensors="pt")

        input_ids = full["input_ids"][0]
        labels = input_ids.clone()
        prompt_len = prompt_ids["input_ids"].shape[1]
        labels[:prompt_len] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": full["attention_mask"][0],
            "labels": labels,
            "pixel_values": full["pixel_values"],
            "image_grid_thw": full["image_grid_thw"],
        }


def collate(batch, pad_token_id):
    max_len = max(item["input_ids"].shape[0] for item in batch)

    def pad(t, value):
        return torch.nn.functional.pad(t, (0, max_len - t.shape[0]), value=value)

    input_ids = torch.stack([pad(b["input_ids"], pad_token_id) for b in batch])
    attention_mask = torch.stack([pad(b["attention_mask"], 0) for b in batch])
    labels = torch.stack([pad(b["labels"], -100) for b in batch])
    pixel_values = torch.cat([b["pixel_values"] for b in batch], dim=0)
    image_grid_thw = torch.cat([b["image_grid_thw"] for b in batch], dim=0)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "pixel_values": pixel_values,
        "image_grid_thw": image_grid_thw,
    }


train_dataset = VQADataset(train_records, processor)
val_dataset = VQADataset(val_records, processor)

# One example through the collator before committing GPU-hours to it.
sample_batch = collate([train_dataset[0], train_dataset[1]], processor.tokenizer.pad_token_id)
for k, v in sample_batch.items():
    print(f"{k:16} {tuple(v.shape)} {v.dtype}")


In [ ]:
from functools import partial

from transformers import Trainer, TrainerCallback, TrainingArguments


class NaNGuard(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        loss = (logs or {}).get("loss")
        if loss is not None and (loss != loss):  # NaN check with no numpy import
            raise RuntimeError(
                f"loss went NaN at step {state.global_step} - lower LEARNING_RATE "
                "further or re-check the LoRA fp32 cast above"
            )


training_args = TrainingArguments(
    output_dir=str(OUT_DIR / "checkpoints"),
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=True,
    bf16=False,                # T4 has no bf16
    max_grad_norm=1.0,         # clip - fp16 + LoRA on Qwen2-VL can NaN otherwise
    optim="paged_adamw_8bit",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50 if SMOKE_TEST else 300,
    save_strategy="epoch",
    save_total_limit=2,
    remove_unused_columns=False,   # Trainer would otherwise drop pixel_values/image_grid_thw
    dataloader_num_workers=2,
    report_to=[],
    label_names=["labels"],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=partial(collate, pad_token_id=processor.tokenizer.pad_token_id),
    callbacks=[NaNGuard()],
)

print(f"steps/epoch ~= {len(train_dataset) // (PER_DEVICE_BATCH * GRAD_ACCUM_STEPS)}")


## Step 6 — train

If `SMOKE_TEST = True` this is ~300 rows / 1 epoch — a few minutes, meant to
prove the collator, masking and step loop don't crash before the real run.
Flip `SMOKE_TEST = False` in the config cell and re-run from there once this
finishes clean.


In [ ]:
import time

start = time.time()
trainer.train()
elapsed = time.time() - start
print(f"training took {elapsed / 60:.1f} min for {NUM_EPOCHS} epoch(s), "
      f"{len(train_dataset)} rows")
if SMOKE_TEST:
    per_epoch = elapsed / NUM_EPOCHS
    print(f"~{per_epoch * 3 / 3600:.1f} GPU-hours estimated for 3 epochs on the "
          f"full {len(train_records)}-row set (rough - smoke set is tiny, "
          f"treat as an order-of-magnitude check against the 30 GPU-hr/week quota)")


In [ ]:
model.save_pretrained(str(ADAPTER_DIR))
processor.save_pretrained(str(ADAPTER_DIR))
print(f"adapter saved -> {ADAPTER_DIR}")
print(sorted(p.name for p in ADAPTER_DIR.iterdir()))


## Step 7 — pin versions

Written from what's actually installed (`pip freeze`), not hand-typed, so
this host and the Mac cannot silently drift apart on the libraries that
read the adapter/merged weights.


In [ ]:
import subprocess

freeze = subprocess.run(["pip", "freeze"], capture_output=True, text=True).stdout
pinned = [
    line for line in freeze.splitlines()
    if any(pkg in line.lower() for pkg in ("transformers", "peft", "accelerate", "torch"))
]
req_path = WORK_DIR / "requirements-train.txt"
req_path.write_text("\n".join(pinned) + "\n")
print(f"wrote {len(pinned)} pins -> {req_path}")
print("\n".join(pinned))


## Step 8 — merge the LoRA into an fp16 base

**You cannot merge a LoRA into a 4-bit base** — `merge_and_unload` needs to
add the low-rank delta directly into the base weight tensors, and a 4-bit
`bnb` tensor isn't a normal addable tensor. Free the 4-bit training model
from GPU memory first, then reload the base fresh in plain fp16.


In [ ]:
import gc
import torch

# Idempotent on purpose: this cell gets re-run after a failure downstream, and
# a plain `del trainer, model` raises NameError the second time - which skips
# empty_cache() and leaves the previous fp16 base resident, so the reload in
# the next cell OOMs on a 14 GB card.
for _name in ("trainer", "model", "base_fp16", "peft_model", "merged"):
    if _name in globals():
        del globals()[_name]

gc.collect()
torch.cuda.empty_cache()
print(f"free VRAM: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB")


In [ ]:
from peft import PeftModel

base_fp16 = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": 0},
    attn_implementation="sdpa",
    token=hf_token,
)

peft_model = PeftModel.from_pretrained(base_fp16, str(ADAPTER_DIR))
for name, param in peft_model.named_parameters():
    if "lora_" in name:
        param.data = param.data.to(torch.float16)  # match the fp16 base for a clean merge

merged = peft_model.merge_and_unload()
merged.save_pretrained(str(MERGED_DIR))
processor.save_pretrained(str(MERGED_DIR))
print(f"merged model saved -> {MERGED_DIR}")
print(sorted(p.name for p in MERGED_DIR.iterdir()))


## Step 9 — convert the merged model to 4-bit MLX

`mlx`/`mlx-vlm` pinned to the **exact** versions the Mac serves with
(`requirements-mac.txt`: `mlx==0.32.2`, `mlx-vlm==0.6.17`) — the whole point
of this step is a directory that `mlx_vlm.load()` on the Mac accepts with no
surprises, and format compatibility is a version-pinning problem just like
`transformers`/`peft` was for the adapter.

If this cell's `pip install` fails (some Kaggle images don't have an MLX
backend that runs off Apple Silicon), don't fight it here: download
`MERGED_DIR` instead — it's a plain fp16 HF model — and run
`python -m mlx_vlm.convert` locally on the Mac, per `scripts/README_LORA.md`
Option B, step 2.


In [ ]:
for _name in ("base_fp16", "peft_model", "merged"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

result = subprocess.run(
    # On Linux `mlx` is a frontend only - the engine ships as a separate
    # backend package selected by extra (mlx-cpu / mlx-cuda-12). On macOS
    # mlx-metal comes in automatically, which is why plain `mlx` works on the
    # Mac and fails here with "libmlx.so: cannot open shared object file".
    # CPU is plenty: conversion is weight loading plus quantisation.
    ["pip", "install", "-q", "mlx[cpu]==0.32.2", "mlx-vlm==0.6.17"],
    capture_output=True, text=True,
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print(result.stderr[-3000:])
    print(
        "\nmlx install failed on this host - download MERGED_DIR (fp16 HF model) "
        "from the Output tab instead and run the conversion locally on the Mac:\n"
        "  python -m mlx_vlm.convert --hf-path merged --mlx-path mlx-4bit -q --q-bits 4"
    )
else:
    print("mlx / mlx-vlm installed")


In [ ]:
convert = subprocess.run(
    [
        "python", "-m", "mlx_vlm.convert",
        "--hf-path", str(MERGED_DIR),
        "--mlx-path", str(MLX_DIR),
        "-q", "--q-bits", "4",
    ],
    capture_output=True, text=True,
)
print(convert.stdout[-3000:])
print(convert.stderr[-3000:])
assert convert.returncode == 0, "mlx_vlm.convert failed - see stderr above"

files = sorted(MLX_DIR.iterdir())
total_gb = sum(f.stat().st_size for f in files if f.is_file()) / 1024**3
print(f"\n{MLX_DIR} -> {len(files)} files, {total_gb:.2f} GB")
for f in files:
    print(" ", f.name)

# This must be a full model, not an adapter - MLXQwen2VL in app/services/vlm.py
# refuses a directory containing adapters.safetensors as an adapter_path.
assert not (MLX_DIR / "adapters.safetensors").exists(), (
    "conversion produced an adapter, not a merged model - something upstream is wrong"
)
assert (MLX_DIR / "config.json").exists(), "no config.json - conversion did not complete"


## Step 10 — package for download

Kaggle's file browser doesn't hand you a directory in one click, so tar it.
Both files land under **Output** on the committed version of this notebook.


In [ ]:
import tarfile

tar_path = WORK_DIR / "mlx-4bit.tar.gz"
with tarfile.open(tar_path, "w:gz") as tar:
    tar.add(MLX_DIR, arcname="mlx-4bit")

print(f"{tar_path}  ({tar_path.stat().st_size / 1024**3:.2f} GB)")
print(f"{WORK_DIR / 'requirements-train.txt'}")


## Done — what to click next

1. If you haven't already: **Save Version → Save & Run All (Commit)**, wait
   for it to finish (check the GPU-hours it used against your weekly 30).
2. Open the committed version's **Output** tab. Download `mlx-4bit.tar.gz`
   and `requirements-train.txt`.
3. On the Mac:

```bash
cd satquery-ai/backend
mkdir -p models && tar -xzf ~/Downloads/mlx-4bit.tar.gz -C models/
SATQUERY_VLM_BACKEND=mlx SATQUERY_MLX_MODEL_ID=$(pwd)/models/mlx-4bit \
    .venv/bin/uvicorn app.main:app
```

4. Compare a few answers against the base `mlx-community` model before
   trusting the fine-tune for the deck — the counting questions are the
   ones this whole exercise was for. `.venv/bin/python -m pytest` should
   still pass unmodified (the test suite talks to the `mock` backend, not
   this model).
5. Keep `requirements-train.txt` next to the repo for the record — if a
   retrain ever answers differently with "no code changes," this file is
   the first thing to diff.
